# Notebook 10e — `10e_social_comparison`

## Overview

**Purpose:** Assemble a side-by-side comparison of the two flood specifications
(3-dimension vs 7-component) for the two social outcomes — unemployment and
poverty — by reading the fit files the four social notebooks save, and export the
comparison to an Excel workbook. Flood is not significant on either social
outcome under either specification, so the choice of specification affects fit,
not conclusions.

**Prerequisite:** run Notebooks 10a, 10b, 10c, 10d first — each writes a `fit_10*.json`.

## Data requirements

| Input | Produced by | Directory |
|---|---|---|
| `fit_10a.json`, `fit_10c.json` | Notebooks 10a (3-dim tpt), 10c (7-var tpt) | `data/processed/` |
| `fit_10b.json`, `fit_10d.json` | Notebooks 10b (3-dim poverty), 10d (7-var poverty) | `data/processed/` |

## Workflow

| Step | Description |
|---|---|
| 1 | Load the four fit files saved by Notebooks 10a-10d |
| 2 | Build the per-outcome comparison tables (criteria × specification, with a `better_model` flag) |
| 3 | Export the comparison to Excel (one sheet per outcome) |

## Outputs

| File | Description |
|---|---|
| `model_comparison_social_3dim_vs_7var.xlsx` | Unemployment and poverty comparison sheets with a `better_model` flag per criterion |

**How to read "better":** lower AIC/BIC/RMSE is better; higher within-R²/adjusted
within-R² is better. BIC is the parsimony-strict criterion. Because flood is not
significant on either social outcome, this comparison informs fit only, not the
substantive conclusion.

---
## STEP 1 — Load the fit files

Read the four `fit_10*.json` files saved by Notebooks 10a-10d.

In [3]:
# ═════════════════════════════════════════════════════════════
# STEP 1 — Load the fit files
# ═════════════════════════════════════════════════════════════
import json, os
import numpy as np, pandas as pd

FIT_DIR  = "../data/processed"
REQUIRED = ["fit_10a.json", "fit_10c.json", "fit_10b.json", "fit_10d.json"]

missing = [f for f in REQUIRED if not os.path.exists(os.path.join(FIT_DIR, f))]
if missing:
    raise FileNotFoundError(f"Missing in {FIT_DIR}: {missing}. Run notebooks 10a-10d first.")

fits = {}
for f in REQUIRED:
    with open(os.path.join(FIT_DIR, f)) as fh:
        d = json.load(fh)
        fits[d["file"]] = d

print("Loaded:", list(fits.keys()))

Loaded: ['10a', '10c', '10b', '10d']


In [5]:
# ═════════════════════════════════════════════════════════════
# STEP 2 — Build the per-outcome comparison tables
# ═════════════════════════════════════════════════════════════
def _better(crit,v3,v7):
    if crit in ("within_R2","adj_within_R2"): return "3-dim" if v3>v7 else ("7-var" if v7>v3 else "tie")
    if crit in ("AIC","BIC","RMSE"):          return "3-dim" if v3<v7 else ("7-var" if v7<v3 else "tie")
    return ""
CRITERIA=["within_R2","adj_within_R2","AIC","BIC","logLik","F_stat","RMSE"]
def build_cmp(d3,d7):
    rows=[]
    for c in CRITERIA:
        v3,v7=d3.get(c),d7.get(c)
        bm=_better(c,v3,v7) if c!="logLik" else ("3-dim" if v3>v7 else "7-var")
        rows.append({"criterion":c,"3-dimension":v3,"7-component":v7,"better_model":bm})
    return pd.DataFrame(rows)

tpt_cmp = build_cmp(fits["10a"], fits["10c"])
pov_cmp = build_cmp(fits["10b"], fits["10d"])
print("=== UNEMPLOYMENT (tpt): 3-dim vs 7-component ===\n")
print(tpt_cmp.to_string(index=False))
print(f"\n  tally: {tpt_cmp['better_model'].value_counts().to_dict()}")
print("\n=== POVERTY: 3-dim vs 7-component ===\n")
print(pov_cmp.to_string(index=False))
print(f"\n  tally: {pov_cmp['better_model'].value_counts().to_dict()}")

=== UNEMPLOYMENT (tpt): 3-dim vs 7-component ===

    criterion  3-dimension  7-component better_model
    within_R2     0.000319     0.000844        7-var
adj_within_R2    -0.000654    -0.000909        3-dim
          AIC -2880.092541 -2876.044059        3-dim
          BIC -2847.368499 -2817.140784        3-dim
       logLik  1445.046270  1447.022029        7-var
       F_stat     2.568983     1.820837             
         RMSE     0.182687     0.182617        7-var

  tally: {'7-var': 3, '3-dim': 3, '': 1}

=== POVERTY: 3-dim vs 7-component ===

    criterion   3-dimension   7-component better_model
    within_R2      0.031302      0.029684        3-dim
adj_within_R2      0.030358      0.027982        3-dim
          AIC -15795.209553 -15790.315795        3-dim
          BIC -15762.485511 -15731.412520        3-dim
       logLik   7902.604777   7904.157898        7-var
       F_stat      2.367155      1.624249             
         RMSE      0.052010      0.051995        7-var

  t

In [7]:
# ═════════════════════════════════════════════════════════════
# STEP 3 — Export to Excel (sheet per outcome)
# ═════════════════════════════════════════════════════════════
OUT="../data/processed/model_comparison_social_3dim_vs_7var.xlsx"
with pd.ExcelWriter(OUT, engine="openpyxl") as xl:
    tpt_cmp.to_excel(xl, sheet_name="1_Unemployment_tpt", index=False)
    pov_cmp.to_excel(xl, sheet_name="2_Poverty", index=False)
print(f"✓ wrote {OUT}")
print("  Sheet 1: Unemployment (tpt) 3dim vs 7var + better_model")
print("  Sheet 2: Poverty 3dim vs 7var + better_model")
print("""
GUIDANCE: as in the growth comparison, BIC (parsimony-strict) is the decisive
criterion; within-R2/AIC tend to favour the richer 7-var spec purely from extra
parameters. Pair with interpretability + collinearity (in 10a-10d). Since flood
is NOT significant on social outcomes under either spec, model choice here mainly
affects fit, not conclusions.""")

✓ wrote ../data/processed/model_comparison_social_3dim_vs_7var.xlsx
  Sheet 1: Unemployment (tpt) 3dim vs 7var + better_model
  Sheet 2: Poverty 3dim vs 7var + better_model

GUIDANCE: as in the growth comparison, BIC (parsimony-strict) is the decisive
criterion; within-R2/AIC tend to favour the richer 7-var spec purely from extra
parameters. Pair with interpretability + collinearity (in 10a-10d). Since flood
is NOT significant on social outcomes under either spec, model choice here mainly
affects fit, not conclusions.
